# ПЗ 5 — Детекция объектов на видео с помощью YOLO

**Задача:** прогнать кадры из ПЗ 2 через YOLOv8, получить bounding boxes и метки объектов, сохранить в JSON.

**Стек:** `ultralytics` (YOLOv8), `opencv-python`, `pandas`

In [ ]:
!pip install ultralytics opencv-python-headless -q

In [ ]:
import os
import json
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm.notebook import tqdm

FRAMES_DIR   = '/content/outputs/frames'
OUTPUT_DIR   = '/content/outputs/detections'
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO('yolov8n.pt')  # nano — быстрая, для GPU можно yolov8s/m
print('YOLO загружена')

In [ ]:
CONF_THRESHOLD = 0.4
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
all_detections = []

for fname in tqdm(frames, desc='YOLO'):
    fpath = os.path.join(FRAMES_DIR, fname)
    results = model(fpath, conf=CONF_THRESHOLD, verbose=False)
    for r in results:
        for box in r.boxes:
            all_detections.append({
                'frame':      fname,
                'class_id':   int(box.cls[0]),
                'class_name': model.names[int(box.cls[0])],
                'confidence': round(float(box.conf[0]), 3),
                'x1': round(float(box.xyxy[0][0])),
                'y1': round(float(box.xyxy[0][1])),
                'x2': round(float(box.xyxy[0][2])),
                'y2': round(float(box.xyxy[0][3])),
            })

df = pd.DataFrame(all_detections)
df.to_csv(f'{OUTPUT_DIR}/yolo_detections.csv', index=False)
print(f'Всего детекций: {len(df)}')
df.head(10)

In [ ]:
# Статистика по классам
print(df.groupby('class_name')['confidence'].agg(['count','mean']).sort_values('count', ascending=False))

In [ ]:
# Визуализация первого кадра с детекциями
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample_frame = frames[0]
img = cv2.imread(os.path.join(FRAMES_DIR, sample_frame))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img_rgb)
frame_dets = df[df['frame'] == sample_frame]
for _, row in frame_dets.iterrows():
    rect = Rectangle((row.x1, row.y1), row.x2-row.x1, row.y2-row.y1,
                     linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.text(row.x1, row.y1-5, f"{row.class_name} {row.confidence:.2f}",
            color='red', fontsize=10)
ax.axis('off')
plt.title(sample_frame)
plt.tight_layout()
plt.show()